In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Cleaning the dataset

In [ ]:
import os, hashlib, random
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DATA_DIR = '/content/drive/MyDrive/IE7615 Deep Learning for AI'  # change this to your dataset location on drive
IMG_SIZE = (224, 224)
BATCH = 8
EXTS = ('.jpg', '.jpeg')

In [ ]:
classes = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
ll=len(classes)
print(len(classes), "folders")
for c in classes[:ll]:
    print(c, len(os.listdir(os.path.join(DATA_DIR, c))))

73 folders
images_OBJ001 100
images_OBJ002 99
images_OBJ003 100
images_OBJ004 100
images_OBJ005 108
images_OBJ006 100
images_OBJ007 100
images_OBJ008 100
images_OBJ009 106
images_OBJ010 100
images_OBJ011 98
images_OBJ012 100
images_OBJ013 100
images_OBJ014 100
images_OBJ015 100
images_OBJ016 100
images_OBJ017 113
images_OBJ018 100
images_OBJ019 100
images_OBJ020 100
images_OBJ021 100
images_OBJ022 99
images_OBJ023 94
images_OBJ024 100
images_OBJ025 100
images_OBJ026 100
images_OBJ027 100
images_OBJ028 100
images_OBJ029 99
images_OBJ030 100
images_OBJ031 100
images_OBJ032 100
images_OBJ033 100
images_OBJ034 100
images_OBJ035 100
images_OBJ036 102
images_OBJ037 100
images_OBJ038 100
images_OBJ039 100
images_OBJ040 100
images_OBJ041 100
images_OBJ042 100
images_OBJ043 100
images_OBJ044 100
images_OBJ045 100
images_OBJ046 100
images_OBJ047 100
images_OBJ048 100
images_OBJ049 105
images_OBJ050 100
images_OBJ051 110
images_OBJ052 100
images_OBJ053 95
images_OBJ054 100
images_OBJ055 100
image

## Removing irrelevant images

In [ ]:
rows = []
bad = []

for c in classes:
    folder = os.path.join(DATA_DIR, c)
    for f in sorted(os.listdir(folder)):
        p = os.path.join(folder, f)

        if c == 'images_OBJ054' and os.path.splitext(f)[0] == 'OBJ054_017':
            bad.append((
                p,
                'Excluded Image which is irrelevant'
            ))
            continue

        if not f.lower().endswith(EXTS):
            continue

        try:
            with Image.open(p) as im:
                im.verify()

            with Image.open(p) as im:
                rows.append((p, c, im.size, im.mode))

        except Exception as e:
            bad.append((p, f'Corrupted or unreadable: {e}'))

df = pd.DataFrame(
    rows,
    columns=['path', 'label', 'size', 'mode']
)

bad_df = pd.DataFrame(bad, columns=['path', 'reason'])

print("\nValid images:", len(df))
print("Excluded files:", len(bad_df))
print("\nImage dimensions:")
print(df['size'].value_counts().head(20))
print("\nColor modes:")
print(df['mode'].value_counts())

display(df.head())
display(bad_df)


Valid images: 7314
Excluded files: 1

Image dimensions:
size
(224, 224)    7314
Name: count, dtype: int64

Color modes:
mode
RGB    7314
Name: count, dtype: int64


,path,label,size,mode
0,/content/drive/MyDrive/IE7615 Deep Learning fo...,images_OBJ001,"(224, 224)",RGB
1,/content/drive/MyDrive/IE7615 Deep Learning fo...,images_OBJ001,"(224, 224)",RGB
2,/content/drive/MyDrive/IE7615 Deep Learning fo...,images_OBJ001,"(224, 224)",RGB
3,/content/drive/MyDrive/IE7615 Deep Learning fo...,images_OBJ001,"(224, 224)",RGB
4,/content/drive/MyDrive/IE7615 Deep Learning fo...,images_OBJ001,"(224, 224)",RGB


,path,reason
0,/content/drive/MyDrive/IE7615 Deep Learning fo...,Excluded Image which is irrelevant


## Check and resize images

In [ ]:
from PIL import Image, ImageOps

wrong_size_mask = df['size'] != (224, 224)
wrong_size_rows = df.loc[wrong_size_mask].copy()

print("Images requiring resizing:", len(wrong_size_rows))

resized_records = []
failed_records = []

for _, row in wrong_size_rows.iterrows():
    path = row['path']
    object_id = str(row['label']).replace('images_', '', 1)
    original_size = row['size']

    try:
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im).convert('RGB')
            resized = im.resize(
                (224, 224),
                Image.Resampling.LANCZOS
            )
            resized.save(path, quality=95)

        df.loc[df['path'] == path, 'size'] = [(224, 224)]
        df.loc[df['path'] == path, 'mode'] = 'RGB'

        resized_records.append({
            'Object ID': object_id,
            'Image': os.path.basename(path),
            'Original Size': original_size,
            'New Size': (224, 224),
            'Path': path
        })

        print(
            f"Resized | {object_id} | "
            f"{os.path.basename(path)} | "
            f"{original_size} -> (224, 224)"
        )

    except Exception as e:
        failed_records.append({
            'Object ID': object_id,
            'Image': os.path.basename(path),
            'Path': path,
            'Error': str(e)
        })
        print(f"FAILED | {object_id} | {path} | {e}")

print("\n--- RESIZING SUMMARY ---")
print("Successfully resized:", len(resized_records))
print("Failed:", len(failed_records))

if resized_records:
    resized_df = pd.DataFrame(resized_records)
    print("\nResized images by Object ID:")
    print(resized_df.groupby('Object ID').size().to_string())
    display(resized_df)

if failed_records:
    print("\nImages that could not be resized:")
    display(pd.DataFrame(failed_records))

remaining = df[df['size'] != (224, 224)]

print("\nImages still not 224x224:", len(remaining))
if not remaining.empty:
    display(remaining[['label', 'path', 'size']])


Images requiring resizing: 0

--- RESIZING SUMMARY ---
Successfully resized: 0
Failed: 0

Images still not 224x224: 0


## Check duplicate images

In [ ]:
import os
import hashlib
from collections import defaultdict

image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

for class_name in sorted(os.listdir(DATA_DIR)):
    class_path = os.path.join(DATA_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    hash_to_files = defaultdict(list)

    for filename in sorted(os.listdir(class_path)):
        if not filename.lower().endswith(image_extensions):
            continue

        file_path = os.path.join(class_path, filename)

        hasher = hashlib.md5()
        try:
            with open(file_path, 'rb') as f:
                for chunk in iter(lambda: f.read(8192), b''):
                    hasher.update(chunk)

            hash_to_files[hasher.hexdigest()].append(filename)

        except (OSError, PermissionError) as e:
            print(f"Could not read {class_name}/{filename}: {e}")

    duplicates = [
        filenames
        for filenames in hash_to_files.values()
        if len(filenames) > 1
    ]

    if duplicates:
        print(f"\n{class_name}:")
        for group in duplicates:
            print("  Duplicate images:", ", ".join(group))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

removal_records = []

for class_name in sorted(os.listdir(DATA_DIR)):
    class_path = os.path.join(DATA_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    hash_to_files = defaultdict(list)

    for filename in sorted(os.listdir(class_path)):
        if not filename.lower().endswith(image_extensions):
            continue

        file_path = os.path.join(class_path, filename)

        try:
            hasher = hashlib.md5()
            with open(file_path, 'rb') as f:
                for chunk in iter(lambda: f.read(8192), b''):
                    hasher.update(chunk)

            hash_to_files[hasher.hexdigest()].append(filename)

        except OSError as e:
            print(f"Could not read {class_name}/{filename}: {e}")

    for filenames in hash_to_files.values():
        if len(filenames) < 2:
            continue

        keep_file = filenames[0]
        keep_path = os.path.join(class_path, keep_file)

        for remove_file in filenames[1:]:
            remove_path = os.path.join(class_path, remove_file)

            fig, axes = plt.subplots(1, 2, figsize=(8, 4))

            with Image.open(keep_path) as im:
                axes[0].imshow(im.copy())
            axes[0].set_title(f"KEEP\n{keep_file}")
            axes[0].axis("off")

            with Image.open(remove_path) as im:
                axes[1].imshow(im.copy())
            axes[1].set_title(f"REMOVE\n{remove_file}")
            axes[1].axis("off")

            fig.suptitle(class_name)
            plt.tight_layout()
            display(fig)
            plt.close(fig)

            os.remove(remove_path)

            removal_records.append(
                (class_name, keep_file, remove_file)
            )

print(f"Total duplicate images removed: {len(removal_records)}")

Total duplicate images removed: 0


# Splitting the dataset

In [ ]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split

DATA_DIR = "/content/drive/MyDrive/IE7615 Deep Learning for AI"
OUTPUT_DIR = "/content/drive/MyDrive/Objectdetection/data/images"

SEED = 42
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

images = []

for class_name in sorted(os.listdir(DATA_DIR)):
    class_path = os.path.join(DATA_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    for filename in os.listdir(class_path):
        if filename.lower().endswith(IMAGE_EXTENSIONS):
            images.append(os.path.join(class_path, filename))

random.seed(SEED)
random.shuffle(images)

print(f"Total images: {len(images)}")

Total images: 7315


In [ ]:
import os
import hashlib
from collections import defaultdict

image_hashes = defaultdict(list)

for class_name in sorted(os.listdir(DATA_DIR)):
    class_path = os.path.join(DATA_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    for filename in os.listdir(class_path):
        if filename.lower().endswith(IMAGE_EXTENSIONS):
            image_path = os.path.join(class_path, filename)

            with open(image_path, "rb") as f:
                file_hash = hashlib.md5(f.read()).hexdigest()

            image_hashes[file_hash].append(image_path)

duplicates = {
    file_hash: paths
    for file_hash, paths in image_hashes.items()
    if len(paths) > 1
}

if duplicates:
    print(f"Found {len(duplicates)} groups of duplicate source images:\n")

    for i, paths in enumerate(duplicates.values(), start=1):
        print(f"Duplicate group {i}:")
        for path in paths:
            print(f"  {path}")
        print()
else:
    print("No duplicate source images found.")

print(f"Total duplicate groups: {len(duplicates)}")
print(f"Total images involved in duplicate groups: {sum(len(paths) for paths in duplicates.values())}")

No duplicate source images found.
Total duplicate groups: 0
Total images involved in duplicate groups: 0


In [ ]:
train_images, temp_images = train_test_split(
    images,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)

val_images, test_images = train_test_split(
    temp_images,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print(f"Train: {len(train_images)} images")
print(f"Val:   {len(val_images)} images")
print(f"Test:  {len(test_images)} images")

Train: 5120 images
Val:   1097 images
Test:  1098 images


In [ ]:
for split in ["train", "val", "test"]:
    split_path = os.path.join(OUTPUT_DIR, split)

    if os.path.exists(split_path):
        shutil.rmtree(split_path)

    os.makedirs(split_path, exist_ok=True)

for split_name, split_images in [
    ("train", train_images),
    ("val", val_images),
    ("test", test_images)
]:
    for image_path in split_images:
        filename = os.path.basename(image_path)
        destination = os.path.join(OUTPUT_DIR, split_name, filename)
        shutil.copy2(image_path, destination)

    print(f"{split_name}: copied {len(split_images)} images")

train: copied 5120 images
val: copied 1097 images
test: copied 1098 images


# Preparing the dataset

In [ ]:
from pathlib import Path
import json

import torch
from PIL import Image, ImageDraw
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
)

In [ ]:
DATA_ROOT = Path(
    "/content/drive/MyDrive/"
    "Objectdetection/data"
)

PROMPTS_PATH = DATA_ROOT.parent / "class_prompts.json"

MODEL_ID = "IDEA-Research/grounding-dino-tiny"

SPLITS = ["train", "val", "test"]

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

BOX_THRESHOLD = 0.25
TEXT_THRESHOLD = 0.25
OVERWRITE_LABELS = True
SAVE_PREVIEWS = True
REVIEW_DIR = DATA_ROOT / "annotation_review"

In [ ]:
import re

if not PROMPTS_PATH.is_file():
    raise FileNotFoundError(
        f"Class prompt file not found: {PROMPTS_PATH}"
    )

with open(PROMPTS_PATH, "r", encoding="utf-8") as file:
    CLASS_PROMPTS = json.load(file)

EXPECTED_CLASSES = sorted({
    path.stem.split("_")[0].upper()
    for split in SPLITS
    for path in (DATA_ROOT / "images" / split).iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
    and re.fullmatch(r"OBJ\d{3}", path.stem.split("_")[0].upper())
})

missing = sorted(set(EXPECTED_CLASSES) - set(CLASS_PROMPTS))
if missing:
    raise ValueError(f"Missing prompts for classes: {missing}")

CLASS_PROMPTS = {name: CLASS_PROMPTS[name] for name in EXPECTED_CLASSES}


for class_name, prompt in CLASS_PROMPTS.items():
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError(
            f"Invalid prompt for {class_name}: {prompt!r}"
        )

CLASS_TO_ID = {
    class_name: index
    for index, class_name in enumerate(EXPECTED_CLASSES)
}

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")
print("Loading Grounding DINO...")

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForZeroShotObjectDetection.from_pretrained(
    MODEL_ID
).to(device)

model.eval()

print("Model loaded successfully.")

Using device: cuda
Loading Grounding DINO...


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  689MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

Model loaded successfully.


In [ ]:
for split in SPLITS:
    label_dir = DATA_ROOT / "labels" / split
    label_dir.mkdir(parents=True, exist_ok=True)

    if SAVE_PREVIEWS:
        (REVIEW_DIR / split).mkdir(
            parents=True,
            exist_ok=True
        )

In [ ]:
total_images = 0
processed_images = 0
detected_images = 0
no_detection_images = 0
background_images = 0
skipped_images = 0
error_images = 0

In [ ]:
for split in SPLITS:

    image_dir = DATA_ROOT / "images" / split
    label_dir = DATA_ROOT / "labels" / split
    preview_dir = REVIEW_DIR / split

    if not image_dir.is_dir():
        raise FileNotFoundError(
            f"Image directory not found: {image_dir}"
        )

    image_paths = sorted(
        path
        for path in image_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

    total_images += len(image_paths)

    print(f"\n{'=' * 60}")
    print(f"Processing split: {split}")
    print(f"Images found: {len(image_paths)}")
    print(f"Labels will be saved to: {label_dir}")
    print(f"{'=' * 60}")

    for index, image_path in enumerate(image_paths, start=1):

        class_name = image_path.stem.split("_")[0].upper()

        if class_name not in CLASS_TO_ID:
            print(
                f"SKIPPED: Unknown class in filename "
                f"{image_path.name}"
            )
            skipped_images += 1
            continue

        class_id = CLASS_TO_ID[class_name]

        label_path = label_dir / f"{image_path.stem}.txt"

        if label_path.exists() and not OVERWRITE_LABELS:
            print(f"SKIPPED existing label: {label_path.name}")
            skipped_images += 1
            continue

        try:
            with Image.open(image_path) as source:
                image = source.convert("RGB")

            image_width, image_height = image.size

            if "_bg_" in image_path.stem.lower():

                label_path.write_text(
                    "",
                    encoding="utf-8"
                )

                background_images += 1
                processed_images += 1

                if SAVE_PREVIEWS:
                    image.save(preview_dir / image_path.name)

                if index % 100 == 0 or index == len(image_paths):
                    print(
                        f"  Progress: {index}/{len(image_paths)}"
                    )

                continue

            prompt = CLASS_PROMPTS[class_name].strip()

            inputs = processor(
                images=image,
                text=prompt,
                return_tensors="pt"
            ).to(device)

            with torch.inference_mode():
                outputs = model(**inputs)

            results = processor.post_process_grounded_object_detection(
                outputs,
                inputs.input_ids,
                threshold=BOX_THRESHOLD,
                text_threshold=TEXT_THRESHOLD,
                target_sizes=[(image_height, image_width)],
            )[0]

            boxes = results["boxes"].detach().cpu()
            scores = results["scores"].detach().cpu()

            label_lines = []
            preview_boxes = []

            for box, score in zip(boxes, scores):

                x1, y1, x2, y2 = [
                    float(value)
                    for value in box.tolist()
                ]

                x1 = max(0.0, min(x1, image_width))
                y1 = max(0.0, min(y1, image_height))
                x2 = max(0.0, min(x2, image_width))
                y2 = max(0.0, min(y2, image_height))

                if x2 <= x1 or y2 <= y1:
                    continue

                x_center = ((x1 + x2) / 2.0) / image_width
                y_center = ((y1 + y2) / 2.0) / image_height

                box_width = (x2 - x1) / image_width
                box_height = (y2 - y1) / image_height

                label_lines.append(
                    f"{class_id} "
                    f"{x_center:.6f} "
                    f"{y_center:.6f} "
                    f"{box_width:.6f} "
                    f"{box_height:.6f}"
                )

                preview_boxes.append(
                    (x1, y1, x2, y2)
                )

            label_path.write_text(
                "\n".join(label_lines)
                + ("\n" if label_lines else ""),
                encoding="utf-8"
            )

            processed_images += 1

            if label_lines:
                detected_images += 1
                status = "DETECTED"
            else:
                no_detection_images += 1
                status = "NO DETECTION - REVIEW"

            if SAVE_PREVIEWS:

                preview = image.copy()
                draw = ImageDraw.Draw(preview)

                for x1, y1, x2, y2 in preview_boxes:

                    draw.rectangle(
                        [x1, y1, x2, y2],
                        outline="red",
                        width=3
                    )

                    text_y = max(0, int(y1) - 18)

                    draw.text(
                        (int(x1), text_y),
                        class_name,
                        fill="red",
                    )

                preview.save(preview_dir / image_path.name)

            if index % 100 == 0 or index == len(image_paths):
                print(
                    f"  Progress: {index}/{len(image_paths)}"
                )

            print(
                f"[{split}] {image_path.name}: "
                f"{len(label_lines)} box(es), {status}"
            )

        except Exception as exc:
            print(
                f"ERROR processing {image_path.name}: {exc}"
            )
            error_images += 1

Streaming output truncated to the last 5000 lines.
[train] OBJ031_005.jpg: 1 box(es), DETECTED
[train] OBJ031_007.jpg: 2 box(es), DETECTED
[train] OBJ031_008.jpg: 1 box(es), DETECTED
[train] OBJ031_009.jpg: 1 box(es), DETECTED
[train] OBJ031_010.jpg: 2 box(es), DETECTED
[train] OBJ031_011.jpg: 1 box(es), DETECTED
[train] OBJ031_012.jpg: 1 box(es), DETECTED
[train] OBJ031_013.jpg: 4 box(es), DETECTED
[train] OBJ031_014.jpg: 1 box(es), DETECTED
[train] OBJ031_015.jpg: 1 box(es), DETECTED
[train] OBJ031_017.jpg: 1 box(es), DETECTED
[train] OBJ031_019.jpg: 1 box(es), DETECTED
[train] OBJ031_021.jpg: 1 box(es), DETECTED
[train] OBJ031_022.jpg: 1 box(es), DETECTED
[train] OBJ031_023.jpg: 1 box(es), DETECTED
[train] OBJ031_024.jpg: 1 box(es), DETECTED
[train] OBJ031_028.jpg: 1 box(es), DETECTED
[train] OBJ031_029.jpg: 1 box(es), DETECTED
[train] OBJ031_030.jpg: 1 box(es), DETECTED
[train] OBJ031_031.jpg: 1 box(es), DETECTED
[train] OBJ031_032.jpg: 1 box(es), DETECTED
[train] OBJ031_034.jpg: 1

In [ ]:
print(f"\n{'=' * 60}")
print("ANNOTATION COMPLETE")
print(f"{'=' * 60}")

print(f"Total images found:       {total_images}")
print(f"Images processed:         {processed_images}")
print(f"Images with detections:   {detected_images}")
print(f"Images with no detection: {no_detection_images}")
print(f"Known background images:  {background_images}")
print(f"Skipped images:           {skipped_images}")
print(f"Images with errors:       {error_images}")

print("\nLabel directories:")

for split in SPLITS:
    print(
        f"  {split}: "
        f"{DATA_ROOT / 'labels' / split}"
    )

if SAVE_PREVIEWS:
    print(f"\nPreview images: {REVIEW_DIR}")


ANNOTATION COMPLETE
Total images found:       7315
Images processed:         7314
Images with detections:   6684
Images with no detection: 263
Known background images:  367
Skipped images:           1
Images with errors:       0

Label directories:
  train: /content/drive/MyDrive/Objectdetection/data/labels/train
  val: /content/drive/MyDrive/Objectdetection/data/labels/val
  test: /content/drive/MyDrive/Objectdetection/data/labels/test

Preview images: /content/drive/MyDrive/Objectdetection/data/annotation_review


In [ ]:
drive.flush_and_unmount()